# Green Fleet Optimizer: SIH 2026 Demo

This notebook runs synthetic data generation, confidence-aware fuel prediction, Pareto optimization, live benchmarking, weather adaptation, bunkering, and retrofit ROI.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
BACKEND = PROJECT_ROOT / 'backend'
sys.path.insert(0, str(BACKEND))
from data.synthetic_generator import run as generate_synthetic_data
RAW = BACKEND / 'data' / 'raw'
PROCESSED = BACKEND / 'data' / 'processed'
if not (RAW / 'vessels.json').exists() or not (PROCESSED / 'voyage_dataset.csv').exists():
    generate_synthetic_data()
voyages = pd.read_csv(PROCESSED / 'voyage_dataset.csv')
print(f'Synthetic voyages: {len(voyages):,}')
voyages.head()

In [ ]:
from prediction.predictor import predict_fuel_consumption_with_uncertainty, calculate_wtw_emissions, calculate_cii_rating
sample = voyages.iloc[0]
prediction = predict_fuel_consumption_with_uncertainty(
    vessel_type=sample.vessel_type, capacity=sample.capacity, engine_power_kw=sample.engine_power_kw,
    distance_nmi=sample.distance_nmi, speed_knots=sample.speed_knots, sea_state=sample.sea_state,
    payload_pct=sample.payload_pct, fuel_type=sample.fuel_type
)
print('Prediction with confidence band:', prediction)
print('CII:', calculate_cii_rating(calculate_wtw_emissions(prediction['fuel_consumption_tonnes'], sample.fuel_type), sample.capacity, sample.distance_nmi))

In [ ]:
from optimization.optimizer import run_fleet_optimization
with open(RAW / 'vessels.json') as handle: vessels = json.load(handle)
with open(RAW / 'routes.json') as handle: routes = json.load(handle)
optimization = run_fleet_optimization(vessels=vessels[:8], routes=routes, algorithm='QIGA', pop_size=12, generations=5, carbon_tax=100, demand_mult=0.8, allowed_fuels=['HFO', 'LNG', 'Methanol', 'Ammonia'], shore_power_enabled=True)
print(optimization['primary_algorithm'], optimization['total_execution_time_seconds'], 'seconds')
pareto = pd.DataFrame(optimization['result']['pareto_front'])
pareto[['cost_usd', 'emissions_tco2e', 'delay_hours']].head()

In [ ]:
from benchmarking.metrics import summarize_algorithm_performance
comparison = run_fleet_optimization(vessels=vessels[:6], routes=routes, algorithm='ALL', pop_size=8, generations=3, carbon_tax=100)
benchmark = pd.DataFrame([summarize_algorithm_performance(result) for result in comparison['all_results'].values()])
benchmark[['algorithm', 'hypervolume_score', 'convergence_generation', 'execution_time_seconds', 'pareto_solutions_count']]

In [ ]:
convergence_rows = []
for name, result in comparison['all_results'].items():
    for generation, history in enumerate(result['convergence_history'], start=1):
        convergence_rows.append({'algorithm': name, 'generation': generation, 'best_cost_usd': history['min_cost']})
convergence = pd.DataFrame(convergence_rows)
convergence

In [ ]:
from api.routes.scenario import WeatherUpdate, weather_reroute, bunkering_recommendations, retrofit_roi
weather = weather_reroute(WeatherUpdate(route_id='R03', new_sea_state=5.5))
print('Weather route:', weather['recommended_route']['id'], '| speed:', weather['recommended_speed_knots'], '| fuel delta:', weather['fuel_delta_tonnes'])
bunkering = bunkering_recommendations(route_id='R03', fuel_type='LNG')
print('Bunker recommendation:', bunkering['recommendation']['port'], '| arbitrage:', bunkering['arbitrage_saving_usd_t'], 'USD/t')
pd.DataFrame(retrofit_roi()['ranked_options'])

In [ ]:
from urllib.request import Request, urlopen
import json as json_lib

print('Jupyter quantum-inspired run:')
print('  algorithm =', optimization['primary_algorithm'])
print('  Pareto solutions =', len(optimization['result']['pareto_front']))
print('  compliance =', optimization['result'].get('compliance_status', 'n/a'))

try:
    with urlopen('http://127.0.0.1:8000/api/health', timeout=3) as response:
        api_health = json_lib.loads(response.read().decode('utf-8'))
    print('Live FastAPI connection:', api_health)
    payload = json_lib.dumps({'algorithm': 'QIGA', 'fleet_size': 4, 'pop_size': 6, 'generations': 2, 'carbon_tax': 100}).encode('utf-8')
    request = Request('http://127.0.0.1:8000/api/optimize-fleet', data=payload, headers={'Content-Type': 'application/json'}, method='POST')
    with urlopen(request, timeout=60) as response:
        live_result = json_lib.loads(response.read().decode('utf-8'))
    print('Live API optimizer:', live_result['primary_algorithm'], '| status:', live_result['status'])
except Exception as error:
    print('Live FastAPI connection unavailable:', error)

print('Data flow: synthetic CSV -> prediction engine -> Jupyter QIGA -> live FastAPI QIGA -> dashboard')